# Multi-Horizon Benchmark Analysis and Visualisation

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

This notebook aggregates results from **all trained models** across **all horizons**  
and generates the publication-quality comparison tables and figures required  
for the research paper.

Run this **after** completing both:
- `01_NiOA_DRNN_Training.ipynb` for every horizon
- `02_Benchmark_Classical_ML.ipynb` and `03_Benchmark_Deep_Learning.ipynb`


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.config import FORECAST_HORIZONS, BENCHMARK_DIR, NIOA_RESULTS_DIR, RESULTS_DIR
from benchmarking.utils.metrics import build_comparison_table

ANALYSIS_DIR = os.path.join(RESULTS_DIR, "analysis")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

sns.set(style="whitegrid", palette="tab10")
print("Analysis notebook ready.")

## 1 · Build Full Results Table

In [ ]:
all_rows = []

for h in FORECAST_HORIZONS:
    try:
        tbl = build_comparison_table(BENCHMARK_DIR, h)
        tbl["Horizon_s"] = h
        all_rows.append(tbl)
    except FileNotFoundError:
        print(f"  No results found for k={h} — skipping.")

results = pd.concat(all_rows, ignore_index=True)

pivot_mae = results.pivot(index="Model", columns="Horizon_s", values="MAE")
print("\nMAE Comparison Table")
print(pivot_mae.to_string())

# Save to CSV
results.to_csv(os.path.join(ANALYSIS_DIR, "full_results.csv"), index=False)
pivot_mae.to_csv(os.path.join(ANALYSIS_DIR, "mae_pivot.csv"))
print(f"\nSaved to : {ANALYSIS_DIR}")

## 2 · MAE vs Horizon Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for model_name, grp in results.groupby("Model"):
    ax.plot(grp["Horizon_s"], grp["MAE"], marker="o", linewidth=2,
            markersize=7, label=model_name)

ax.set_xscale("log")
ax.set_xticks(FORECAST_HORIZONS)
ax.set_xticklabels([str(h) for h in FORECAST_HORIZONS])
ax.set_xlabel("Forecast Horizon k (seconds)")
ax.set_ylabel("MAE (kWh)")
ax.set_title("MAE vs Forecast Horizon — All Models")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(ANALYSIS_DIR, "mae_vs_horizon.png"), dpi=200)
plt.show()
plt.close()

## 3 · R² vs Horizon Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for model_name, grp in results.groupby("Model"):
    ax.plot(grp["Horizon_s"], grp["R2"], marker="s", linewidth=2,
            markersize=7, label=model_name)

ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xscale("log")
ax.set_xticks(FORECAST_HORIZONS)
ax.set_xticklabels([str(h) for h in FORECAST_HORIZONS])
ax.set_xlabel("Forecast Horizon k (seconds)")
ax.set_ylabel("R²")
ax.set_title("R² vs Forecast Horizon — All Models")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(ANALYSIS_DIR, "r2_vs_horizon.png"), dpi=200)
plt.show()
plt.close()

## 4 · Per-Horizon Grouped Bar Chart (MAE)

In [ ]:
available_horizons = results["Horizon_s"].unique()
n_h = len(available_horizons)
fig, axes = plt.subplots(1, n_h, figsize=(5 * n_h, 5), sharey=False)

if n_h == 1:
    axes = [axes]

for ax, h in zip(axes, sorted(available_horizons)):
    sub = results[results["Horizon_s"] == h].sort_values("MAE")
    bars = ax.barh(sub["Model"], sub["MAE"], color=sns.color_palette("tab10", len(sub)))
    ax.set_title(f"k = {h} s")
    ax.set_xlabel("MAE")
    ax.invert_yaxis()

plt.suptitle("MAE Comparison Across All Horizons", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(ANALYSIS_DIR, "mae_bar_all_horizons.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close()

print(f"All analysis figures saved to : {ANALYSIS_DIR}")